<a href="https://colab.research.google.com/github/pegumzs/SSD_MVP2/blob/main/MVP2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MVP – Sistema de Suporte à Decisão para Implantação de Data Center em Nuvem

## Contexto
Empresas responsáveis por armazenamento de dados precisam avaliar fatores operacionais antes de expandir sua infraestrutura. O desempenho dos servidores afeta diretamente os custos operacionais (energia e hardware) e os riscos de inatividade.

## Objetivo
Desenvolver um MVP de um Sistema de Suporte à Decisão para avaliar a viabilidade de uma nova unidade de data center.

O sistema traduz métricas técnicas da base "Cloud Computing Performance Metrics" do Kaggle em três dimensões de negócio: **Custo, Tempo e Risco**.

In [3]:
!pip install -q "kagglehub[pandas-datasets]"

In [9]:
import pandas as pd
import os
import glob
import kagglehub

# Faz o download da base de dados e salva o caminho da pasta
caminho = kagglehub.dataset_download(
    "abdurraziq01/cloud-computing-performance-metrics"
)

# Busca automaticamente qualquer arquivo .csv dentro dessa pasta
arquivos_csv = glob.glob(
    os.path.join(caminho, "**", "*.csv"),
    recursive=True
)

# Carrega o arquivo encontrado para o DataFrame
df_raw = pd.read_csv(arquivos_csv[0])

print("Base original carregada com sucesso!")
display(df_raw.head())

Using Colab cache for faster access to the 'cloud-computing-performance-metrics' dataset.
Base original carregada com sucesso!


,vm_id,timestamp,cpu_usage,memory_usage,network_traffic,power_consumption,num_executed_instructions,execution_time,energy_efficiency,task_type,task_priority,task_status
0,c5215826-6237-4a33-9312-72c1df909881,2023-01-25 09:10:54,54.881350,78.950861,164.775973,287.808986,7527.0,69.345575,0.553589,network,medium,waiting
1,29690bc6-1f34-403b-b509-a1ecb1834fb8,2023-01-26 04:46:34,71.518937,29.901883,NaN,362.273569,5348.0,41.396040,0.349856,io,high,completed
2,2e55abc3-5bad-46cb-b445-a577f5e9bf2a,2023-01-13 23:39:47,NaN,92.709195,203.674847,231.467903,5483.0,24.602549,0.796277,io,medium,completed
3,e672e32f-c134-4fbc-992b-34eb63bef6bf,2023-02-09 11:45:49,54.488318,88.100960,NaN,195.639954,5876.0,16.456670,0.529511,compute,high,completed
4,f38b8b50-6926-4533-be4f-89ad11624071,2023-06-14 08:27:26,42.365480,NaN,NaN,359.451537,3361.0,55.307992,0.351907,NaN,medium,waiting


In [10]:
print("Nomes exatos das colunas:", df_raw.columns.tolist())

Nomes exatos das colunas: ['vm_id', 'timestamp', 'cpu_usage', 'memory_usage', 'network_traffic', 'power_consumption', 'num_executed_instructions', 'execution_time', 'energy_efficiency', 'task_type', 'task_priority', 'task_status']


In [14]:
# Célula 4 - Feature Engineering (Mapeamento das Variáveis Reais)

df = pd.DataFrame()
df['Project_ID'] = df_raw['vm_id'] # Usando o ID da máquina virtual como ID do projeto

# 1. Custo (Baseado no consumo de recursos: CPU e Memória)
df['Estimated_Cost_USD'] = (df_raw['cpu_usage'] * 1000) + (df_raw['memory_usage'] * 500)

# 2. Risco (Baseado no tráfego de rede e consumo de energia)
df['Risk_Assessment_Score'] = (df_raw['network_traffic'] * 0.6) + (df_raw['power_consumption'] * 0.4)

# 3. Tempo de Setup (Estimado com base no tempo de execução na nuvem)
df['Time_Estimate_Days'] = df_raw['execution_time'] * 2

print("Dimensão da base processada para o SSD:", df.shape)
display(df.head())

Dimensão da base processada para o SSD: (2000000, 4)


,Project_ID,Estimated_Cost_USD,Risk_Assessment_Score,Time_Estimate_Days
0,c5215826-6237-4a33-9312-72c1df909881,94356.780902,213.989178,138.691150
1,29690bc6-1f34-403b-b509-a1ecb1834fb8,86469.877932,NaN,82.792079
2,2e55abc3-5bad-46cb-b445-a577f5e9bf2a,NaN,214.792069,49.205098
3,e672e32f-c134-4fbc-992b-34eb63bef6bf,98538.798114,NaN,32.913340
4,f38b8b50-6926-4533-be4f-89ad11624071,NaN,NaN,110.615983


## Benchmark histórico
A base será utilizada para estabelecer referências de custo, risco e prazo. Serão analisados os quartis para posicionar o projeto da nova unidade de data center em relação ao histórico de carga e desempenho.

In [15]:
# Variáveis centrais para análise da nova unidade
variaveis_principais = [
    "Estimated_Cost_USD",
    "Risk_Assessment_Score",
    "Time_Estimate_Days"
]

benchmark = df[variaveis_principais].describe(
    percentiles=[0.25, 0.50, 0.75]
).round(2)

display(benchmark)

,Estimated_Cost_USD,Risk_Assessment_Score,Time_Estimate_Days
count,1620055.00,1620193.00,1800173.00
mean,75005.95,399.99,99.94
std,32262.48,182.51,57.72
min,128.73,0.23,0.00
25%,50003.60,250.21,49.96
50%,75036.69,400.00,99.93
75%,99975.40,549.90,149.95
max,149868.46,799.72,200.00


In [16]:
def calcular_percentil(valor, serie):
    return (serie <= valor).mean() * 100

## MVP — Avaliação de uma Nova Unidade de Data Center

O sistema recebe três informações do projeto e as compara com a base histórica, convertendo-as em um percentil.

Diferente de uma média simples, este SSD utiliza uma **abordagem multicritério (AHP)**, atribuindo pesos diferentes às dimensões, uma vez que em data centers o custo de infraestrutura é o fator mais crítico:
- **Custo:** Peso 50%
- **Risco:** Peso 30%
- **Tempo:** Peso 20%

In [17]:
def avaliar_nova_unidade(custo_usd, risco, tempo_dias):

    # Posicionamento do projeto
    score_custo = calcular_percentil(custo_usd, df["Estimated_Cost_USD"])
    score_risco = calcular_percentil(risco, df["Risk_Assessment_Score"])
    score_tempo = calcular_percentil(tempo_dias, df["Time_Estimate_Days"])

    # Índice geral multicritério (AHP)
    indice_geral = (score_custo * 0.50) + (score_risco * 0.30) + (score_tempo * 0.20)

    print("=== SISTEMA DE SUPORTE À DECISÃO ===")
    print(f"Custo estimado projetado: US$ {custo_usd:,.2f}")
    print(f"Risco de gargalo projetado: {risco:.1f}/100")
    print(f"Prazo estimado de setup: {tempo_dias:.0f} dias")

    print("\n=== POSIÇÃO EM RELAÇÃO AO HISTÓRICO DE INFRAESTRUTURA ===")
    print(f"Exposição de custo: {score_custo:.1f}%")
    print(f"Exposição de risco: {score_risco:.1f}%")
    print(f"Exposição de prazo: {score_tempo:.1f}%")
    print(f"Índice geral ponderado: {indice_geral:.1f}%")

    # Classificação geral
    if indice_geral < 33:
        situacao = "FAVORÁVEL"
    elif indice_geral < 67:
        situacao = "ATENÇÃO"
    else:
        situacao = "CRÍTICA"

    print(f"\nSituação do projeto: {situacao}")

    print("\n=== RECOMENDAÇÕES ESTRATÉGICAS ===")

    if score_custo >= 67:
        print("- CUSTO: Rever a arquitetura de servidores. Aplicar técnicas de eficiência térmica/energética para baixar o OPEX.")

    if score_risco >= 67:
        print("- RISCO: Latência e instabilidade acima da média. Necessário priorizar redundância na topologia de rede.")

    if score_tempo >= 67:
        print("- TEMPO: Setup lento. Avaliar implantação em fases (faseamento) para não travar a operação.")

    if situacao == "FAVORÁVEL":
        print("\n-> O projeto é viável e os parâmetros estão otimizados. Avançar para execução.")
    elif situacao == "ATENÇÃO":
        print("\n-> Viabilidade parcial. Ajustar o escopo do projeto focando no indicador com maior exposição.")
    else:
        print("\n-> INVIÁVEL. A configuração atual sobrecarregará a operação. Refazer o planejamento da unidade.")

## Teste do MVP
Simulando a implantação de uma unidade de alta performance (Custo: 85.000, Risco: 65, Tempo: 45 dias).

In [18]:
avaliar_nova_unidade(
    custo_usd=85000,
    risco=65,
    tempo_dias=45
)

=== SISTEMA DE SUPORTE À DECISÃO ===
Custo estimado projetado: US$ 85,000.00
Risco de gargalo projetado: 65.0/100
Prazo estimado de setup: 45 dias

=== POSIÇÃO EM RELAÇÃO AO HISTÓRICO DE INFRAESTRUTURA ===
Exposição de custo: 48.6%
Exposição de risco: 1.4%
Exposição de prazo: 20.3%
Índice geral ponderado: 28.8%

Situação do projeto: FAVORÁVEL

=== RECOMENDAÇÕES ESTRATÉGICAS ===

-> O projeto é viável e os parâmetros estão otimizados. Avançar para execução.
